https://sparkbyexamples.com/pyspark/pyspark-join-explained-with-examples/

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.master("local[1]") \
    .appName("SparkByExamples.com") \
    .getOrCreate()

In [4]:
emp = [(1,"Smith",-1,"2018","10","M",3000), \
    (2,"Rose",1,"2010","20","M",4000), \
    (3,"Williams",1,"2010","10","M",1000), \
    (4,"Jones",2,"2005","10","F",2000), \
    (5,"Brown",2,"2010","40","",-1), \
      (6,"Brown",2,"2010","50","",-1) \
  ]
empColumns = ["emp_id","name","superior_emp_id","year_joined", \
       "emp_dept_id","gender","salary"]

empDF = spark.createDataFrame(data=emp, schema = empColumns)
empDF.printSchema()
empDF.show(truncate=False)

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: long (nullable = true)
 |-- year_joined: string (nullable = true)
 |-- emp_dept_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)

+------+--------+---------------+-----------+-----------+------+------+
|emp_id|name    |superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+--------+---------------+-----------+-----------+------+------+
|1     |Smith   |-1             |2018       |10         |M     |3000  |
|2     |Rose    |1              |2010       |20         |M     |4000  |
|3     |Williams|1              |2010       |10         |M     |1000  |
|4     |Jones   |2              |2005       |10         |F     |2000  |
|5     |Brown   |2              |2010       |40         |      |-1    |
|6     |Brown   |2              |2010       |50         |      |-1    |
+------+--------+---------------+-----------+-----------+------+-----

In [5]:
dept = [("Finance",10), \
    ("Marketing",20), \
    ("Sales",30), \
    ("IT",40) \
  ]
deptColumns = ["dept_name","dept_id"]
deptDF = spark.createDataFrame(data=dept, schema = deptColumns)
deptDF.printSchema()
deptDF.show(truncate=False)

root
 |-- dept_name: string (nullable = true)
 |-- dept_id: long (nullable = true)

+---------+-------+
|dept_name|dept_id|
+---------+-------+
|Finance  |10     |
|Marketing|20     |
|Sales    |30     |
|IT       |40     |
+---------+-------+



## Inner join
Columns from left and right for rows with matching 'on' keys

In [6]:
empDF.join(deptDF, empDF.emp_dept_id == deptDF.dept_id, "inner").show(truncate=False)

+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name    |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|1     |Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|3     |Williams|1              |2010       |10         |M     |1000  |Finance  |10     |
|4     |Jones   |2              |2005       |10         |F     |2000  |Finance  |10     |
|2     |Rose    |1              |2010       |20         |M     |4000  |Marketing|20     |
|5     |Brown   |2              |2010       |40         |      |-1    |IT       |40     |
+------+--------+---------------+-----------+-----------+------+------+---------+-------+



Above, there is one less row than empDF, because the second 'Brown' had an emp_dept_id of 50 which doesn't appear in deptDF.
Likewise, there is no row with 30 dept_id because there is no row with emp_dept_id of 30 in empDF

## Full outer join
Join all columns from both DFs inclusive of all rows

In [7]:
empDF.join(deptDF,
           empDF.emp_dept_id == deptDF.dept_id, # On is redundant for full outer join
           "outer").show(truncate=False)

empDF.join(deptDF,
           empDF.emp_dept_id == deptDF.dept_id, # On is redundant for full outer join
           "full").show(truncate=False)

empDF.join(deptDF,
           empDF.emp_dept_id == deptDF.dept_id, # On is redundant for full outer join
           "fullouter").show(truncate=False)

+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name    |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|6     |Brown   |2              |2010       |50         |      |-1    |null     |null   |
|1     |Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|3     |Williams|1              |2010       |10         |M     |1000  |Finance  |10     |
|4     |Jones   |2              |2005       |10         |F     |2000  |Finance  |10     |
|null  |null    |null           |null       |null       |null  |null  |Sales    |30     |
|2     |Rose    |1              |2010       |20         |M     |4000  |Marketing|20     |
|5     |Brown   |2              |2010       |40         |      |-1    |IT       |40     |
+------+--------+---------------+-----------+-----------+------+------+---------+-------+

+------+-

## Left Outer Join
Left columns for all, right columns for rows with matches in 'on'; All left rows; matching right rows

In [8]:
empDF.join(deptDF, empDF.emp_dept_id == deptDF.dept_id, "left").show(truncate=False)
empDF.join(deptDF, empDF.emp_dept_id == deptDF.dept_id, "leftouter").show(truncate=False)

+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name    |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|6     |Brown   |2              |2010       |50         |      |-1    |null     |null   |
|1     |Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|3     |Williams|1              |2010       |10         |M     |1000  |Finance  |10     |
|4     |Jones   |2              |2005       |10         |F     |2000  |Finance  |10     |
|2     |Rose    |1              |2010       |20         |M     |4000  |Marketing|20     |
|5     |Brown   |2              |2010       |40         |      |-1    |IT       |40     |
+------+--------+---------------+-----------+-----------+------+------+---------+-------+

+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|n

Above we have emp_dept_id of 50 because it appears in empDF (left);
Note the null dept_name and dept_id for this row because there are no corresponding values on the right.

We still don't have dept_id of 30 because it appears only in right and not in left.

A right/rightouter would have been the reverse.

## Left Semi Join
Returns rows of the left for columns which have a match in the "on" column(s). Result has columns from the left only.
(Skipping right variations of left joins we've covered - right does the same only swap left and right; the same can be achieved by swapping arguments)

In [9]:
empDF.join(deptDF, empDF.emp_dept_id == deptDF.dept_id, "left_semi").show(truncate=False)

+------+--------+---------------+-----------+-----------+------+------+
|emp_id|name    |superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+--------+---------------+-----------+-----------+------+------+
|1     |Smith   |-1             |2018       |10         |M     |3000  |
|3     |Williams|1              |2010       |10         |M     |1000  |
|4     |Jones   |2              |2005       |10         |F     |2000  |
|2     |Rose    |1              |2010       |20         |M     |4000  |
|5     |Brown   |2              |2010       |40         |      |-1    |
+------+--------+---------------+-----------+-----------+------+------+



Above, the result contains neither 30 nor 50 since there was not a match on those columns; note the absence of the columns from the right table (result does not have dept_name nor dept_id). This join may be useful in matching operations, where we want to keep data from one DF only if it exists in the other, analagous to the set union operation A U B

## Left Anti Join
Returns rows from the left which do not have a match in the "on" column(s). Result has columns from the left only.


In [10]:
empDF.join(deptDF, empDF.emp_dept_id == deptDF.dept_id, "left_anti").show(truncate=False)

+------+-----+---------------+-----------+-----------+------+------+
|emp_id|name |superior_emp_id|year_joined|emp_dept_id|gender|salary|
+------+-----+---------------+-----------+-----------+------+------+
|6     |Brown|2              |2010       |50         |      |-1    |
+------+-----+---------------+-----------+-----------+------+------+



Above we have the row containing only 50 and not 30. This because 30 is a mismatch which appears on the _right_ and thus is excluded.
This join may be thought of as a one-sided diff, analagous to the set difference operation A-B

## Self Join
You can specifiy the df itself as the first argument of any of the joins above, thus joining it to itself, though you may have to alias it in order to do anything particularly useful, as explained below.

In [16]:
from pyspark.sql.functions import col

empDF.alias("emp1").join(empDF.alias("emp2"), \
    col("emp1.superior_emp_id") == col("emp2.emp_id"),"inner") \
    .select(col("emp1.emp_id"),col("emp1.name"), \
      col("emp2.emp_id").alias("superior_emp_id"), \
      col("emp2.name").alias("superior_emp_name")) \
   .show(truncate=False)


+------+--------+---------------+-----------------+
|emp_id|name    |superior_emp_id|superior_emp_name|
+------+--------+---------------+-----------------+
|2     |Rose    |1              |Smith            |
|3     |Williams|1              |Smith            |
|4     |Jones   |2              |Rose             |
|5     |Brown   |2              |Rose             |
|6     |Brown   |2              |Rose             |
+------+--------+---------------+-----------------+



The above looks up each employee's superior's name by joining empDF to itself on superior_emp_id == emp_id; aliasing the DFs is necessary first in the ambiguity of the join 'on' condition - "superior_emp_id" and "emp_id" appear in both and it needs to be clear that we're joining on the left's superior_emp_id and the right's emp_id.
Then secondly, in order to select the emp_id and name from the left and the superior_emp_id and superior_emp_name (emp2.name) from the right.

## Cross join
Danger

Slow and memory-intensive

Avoid

Number of row results equal to the cartesian product of the two Dataframes,

i.e., crossjoin pairs every row on the left to every row on the right. A x B

Result includes columns from both.

In [35]:
forbidden_fruit = empDF.crossJoin(deptDF)
forbidden_fruit.show(truncate=False)

print(empDF.count())
print(deptDF.count())
print(forbidden_fruit.count())

+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name    |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|1     |Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|1     |Smith   |-1             |2018       |10         |M     |3000  |Marketing|20     |
|1     |Smith   |-1             |2018       |10         |M     |3000  |Sales    |30     |
|1     |Smith   |-1             |2018       |10         |M     |3000  |IT       |40     |
|2     |Rose    |1              |2010       |20         |M     |4000  |Finance  |10     |
|2     |Rose    |1              |2010       |20         |M     |4000  |Marketing|20     |
|2     |Rose    |1              |2010       |20         |M     |4000  |Sales    |30     |
|2     |Rose    |1              |2010       |20         |M     |4000  |IT       |40     |
|3     |Wi



This may be limited by the on condition, where it will be all matching rows on the left x all matching rows on the right

In [42]:
forbidden_twot = empDF.join(deptDF, ((empDF.superior_emp_id > 1) & (deptDF.dept_id % 20 == 0)), "cross")
forbidden_twot.show(truncate=False)

print(empDF.count())
print(empDF.where(empDF.superior_emp_id > 1).count())
print()

print(deptDF.count())
print(deptDF.where(deptDF.dept_id % 20 == 0).count())
print()

print(forbidden_twot.count())

+------+-----+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+-----+---------------+-----------+-----------+------+------+---------+-------+
|4     |Jones|2              |2005       |10         |F     |2000  |Marketing|20     |
|4     |Jones|2              |2005       |10         |F     |2000  |IT       |40     |
|5     |Brown|2              |2010       |40         |      |-1    |Marketing|20     |
|5     |Brown|2              |2010       |40         |      |-1    |IT       |40     |
|6     |Brown|2              |2010       |50         |      |-1    |Marketing|20     |
|6     |Brown|2              |2010       |50         |      |-1    |IT       |40     |
+------+-----+---------------+-----------+-----------+------+------+---------+-------+

6
3

4
2

6


If cross join is used with an 'on' condition comparing the two dataframes, (with one match between them for each), the results will be exaclty the same as an inner join

## Join using SQL Expressions

Here using temp views in place of tables for demonstration purposes

In [43]:
empDF.createOrReplaceTempView("EMP")
deptDF.createOrReplaceTempView("DEPT")

In [45]:
joinDF = spark.sql("select * from EMP e, DEPT d where e.emp_dept_id == d.dept_id") # Implicit inner join
joinDF.show(truncate=False)

+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name    |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|1     |Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|3     |Williams|1              |2010       |10         |M     |1000  |Finance  |10     |
|4     |Jones   |2              |2005       |10         |F     |2000  |Finance  |10     |
|2     |Rose    |1              |2010       |20         |M     |4000  |Marketing|20     |
|5     |Brown   |2              |2010       |40         |      |-1    |IT       |40     |
+------+--------+---------------+-----------+-----------+------+------+---------+-------+



In [48]:
joinDF2 = spark.sql("select * from EMP e INNER JOIN DEPT d ON e.emp_dept_id == d.dept_id") # Explicit inner join
joinDF2.show(truncate=False)

+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|emp_id|name    |superior_emp_id|year_joined|emp_dept_id|gender|salary|dept_name|dept_id|
+------+--------+---------------+-----------+-----------+------+------+---------+-------+
|1     |Smith   |-1             |2018       |10         |M     |3000  |Finance  |10     |
|3     |Williams|1              |2010       |10         |M     |1000  |Finance  |10     |
|4     |Jones   |2              |2005       |10         |F     |2000  |Finance  |10     |
|2     |Rose    |1              |2010       |20         |M     |4000  |Marketing|20     |
|5     |Brown   |2              |2010       |40         |      |-1    |IT       |40     |
+------+--------+---------------+-----------+-----------+------+------+---------+-------+



Any legal sql join is fair play in this way.